# 🧠 Supply Chain & Logistics Analytics: 02. Explainable Delivery Delay Prediction

**Author:** Senior Data Scientist & ML Engineer  
**Role Target:** Data Science / Machine Learning Internship Interview Showcase  
**Dataset:** Processed Logistics Dataset (6,500 records)  
**Primary Objective:** Build, evaluate, interpret, and explain supervised classification models (Logistic Regression vs. Decision Tree) to predict whether a dispatched shipment will arrive **On Time (0)** or **Delayed (1)**, adhering to strict data leakage prevention.

---

### 🎓 Interview Philosophy: Explainability Over "Black-Box" Complexity
In commercial enterprise logistics, a 1% higher accuracy from a deep neural network is useless if supply chain managers cannot understand **why** a delay was predicted.
This notebook demonstrates **white-box, explainable machine learning**:
- **Logistic Regression:** Linear log-odds interpretability via coefficients and odds ratios.
- **Decision Tree:** Intuitive, deterministic if-then business rules with visual decision path tracking.

---
## 🛠️ Step 0: Imports, Environment Setup & Seeds

### 1. WHAT We Are Doing
We import Scikit-Learn estimators, transformers, evaluation metrics, and visualization tools. We establish fixed random seeds (`random_state=42`) for full determinism and reproducibility.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Scikit-Learn Model Selection & Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('deep')

print("All ML libraries and modules loaded successfully.")

All ML libraries and modules loaded successfully.


---
## 🛡️ Step 1: Data Ingestion & Target Leakage Prevention Audit

### 1. WHAT We Are Doing
We load `data/processed/cleaned_delivery_data.csv` and explicitly isolate:
1. **Prediction Features ($X$):** Only features known **at or before dispatch**.
2. **Target Variable ($y$):** `Delivery_Delayed` (0 = On Time, 1 = Delayed).
3. **Excluded Post-Delivery Leakage Columns:** `Delivery_Date`, `Delivery_Status`, `Delivery_Delay_Days`, `Actual_Transit_Days`.

### 2. WHY We Are Doing It (Critical Interview Concept)
**Target Leakage** occurs when data outside the operational scope of the model at inference time is included during training.
- If we include `Delivery_Delay_Days`, any model learns the trivial rule: `if Delay_Days > 0 -> Delayed`.
- In production, at 8:00 AM when the truck leaves the dock, `Delivery_Delay_Days` is literally unknown. Training on it produces fraudulent metric scores that crash in real-world deployment.

### 3. Interview Q&A
> **Interviewer:** *How do you identify and prevent target leakage in a logistics delay prediction pipeline?*  
> **Candidate Answer:** *I establish a strict **Operational Time Horizon**. I list every feature along with its timestamp of creation. Any feature generated after the prediction event (e.g., after the truck leaves the dock) is quarantined into an analytical post-delivery table and completely excluded from model training ($X$).*

In [2]:
data_path = '../data/processed/cleaned_delivery_data.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/cleaned_delivery_data.csv'

df = pd.read_csv(data_path)
print(f"Loaded processed dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Explicit Feature Segregation
target_col = 'Delivery_Delayed'

# Leakage columns to drop from feature matrix X
leakage_and_id_cols = [
    'Shipment_ID',           # Unique identifier (no predictive signal)
    'Order_Date',            # Raw date string
    'Dispatch_Date',         # Raw date string
    'Delivery_Date',         # LEAKAGE: Occurs after prediction point
    'Delivery_Status',       # LEAKAGE: Direct text label of target
    'Delivery_Delay_Days',   # LEAKAGE: Direct magnitude of target
    'Actual_Transit_Days',   # LEAKAGE: Requires Delivery_Date
    'Order_Year_Month',      # High-cardinality metadata
    'Delivery_Delayed'       # Target variable
]

# Valid pre-transit / at-dispatch feature columns
feature_cols = [col for col in df.columns if col not in leakage_and_id_cols]

X = df[feature_cols]
y = df[target_col]

print(f"\nTarget Variable: '{target_col}' (Class balance: {y.mean()*100:.2f}% delayed)")
print(f"Operational Feature Matrix ({X.shape[1]} features):\n{list(X.columns)}")

Loaded processed dataset: 6,500 rows × 24 columns

Target Variable: 'Delivery_Delayed' (Class balance: 44.98% delayed)
Operational Feature Matrix (15 features):
['Origin', 'Destination', 'Distance', 'Shipping_Mode', 'Weather_Condition', 'Traffic_Level', 'Waiting_Time', 'Transportation_Cost', 'Dispatch_Latency_Days', 'Cost_Per_KM', 'Distance_Tier', 'Order_Day_Of_Week', 'Order_Month', 'Is_Weekend_Order', 'High_Waiting_Flag']


---
## ⚙️ Step 2: Preprocessing Pipeline & Stratified Train-Test Split

### 1. WHAT We Are Doing
1. **Stratified Split (80% Train, 20% Test):** Preserves the exact $55:45$ On-Time to Delayed proportion in both splits.
2. **ColumnTransformer:**
   - **Numerical Features** (`Distance`, `Waiting_Time`, `Transportation_Cost`, `Dispatch_Latency_Days`, `Cost_Per_KM`): Scaled via `StandardScaler()`:
     $$z = \frac{x - \mu}{\sigma}$$
   - **Categorical Features** (`Origin`, `Destination`, `Shipping_Mode`, `Weather_Condition`, `Traffic_Level`, `Distance_Tier`, `Order_Day_Of_Week`): Encoded via `OneHotEncoder(handle_unknown='ignore')`.

### 2. WHY We Are Doing It
- `StandardScaler` ensures distance features (measured in thousands of km) do not numerically dominate dock waiting times (measured in hours) during Logistic Regression gradient updates.
- Fitting transformers **strictly on `X_train`** and only transforming `X_test` avoids **data snooping / distribution leakage**.

In [3]:
# Identify Numerical and Categorical feature columns
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTraining Set: {X_train.shape[0]:,} samples | Delayed Class: {y_train.mean()*100:.2f}%")
print(f"Test Set:     {X_test.shape[0]:,} samples | Delayed Class: {y_test.mean()*100:.2f}%")

# Construct Scikit-Learn Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

# Fit preprocessor on training data only
preprocessor.fit(X_train)

# Extract encoded feature names for explainability
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_feature_names = num_features + encoded_cat_names

print(f"Total processed feature dimensions after One-Hot Encoding: {len(all_feature_names)}")

Numerical features (8): ['Distance', 'Waiting_Time', 'Transportation_Cost', 'Dispatch_Latency_Days', 'Cost_Per_KM', 'Order_Month', 'Is_Weekend_Order', 'High_Waiting_Flag']
Categorical features (7): ['Origin', 'Destination', 'Shipping_Mode', 'Weather_Condition', 'Traffic_Level', 'Distance_Tier', 'Order_Day_Of_Week']

Training Set: 5,200 samples | Delayed Class: 44.98%
Test Set:     1,300 samples | Delayed Class: 45.00%
Total processed feature dimensions after One-Hot Encoding: 36


---
## 📈 Step 3: Model 1 — Logistic Regression Deep Dive

### 1. WHAT We Are Doing
We train a binary **Logistic Regression** classifier with $L_2$ regularization (`C=1.0`) on the preprocessed training dataset, extract learned coefficients, and analyze odds ratios.

### 2. HOW It Works (Mathematical Rigor)
Logistic regression computes a linear score ($z$):
$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$$
This score is passed through the **Sigmoid (Logistic) Function** to map real numbers $(-\infty, \infty)$ into a valid probability $p \in (0, 1)$:
$$p(y=1|X) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

#### The Log-Odds (Logit) Formulation:
$$\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$$
- **Odds Ratio ($e^{\beta_i}$):** For a one-unit increase in standardized feature $x_i$, the odds of delay multiply by $e^{\beta_i}$.
  - If $\beta_i > 0 \implies e^{\beta_i} > 1$: Feature **increases** delay probability.
  - If $\beta_i < 0 \implies e^{\beta_i} < 1$: Feature **decreases** delay probability (promotes on-time delivery).

### 3. Interview Q&A
> **Interviewer:** *Why do we use the Sigmoid function instead of a linear step function?*  
> **Candidate Answer:** *A linear step function is non-differentiable (gradient is zero everywhere except at the step), which prevents gradient-based optimization like gradient descent. The Sigmoid function is smooth, continuously differentiable, and maps the linear combination into a calibrated probability between 0 and 1.*

In [4]:
# Define Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

# Train Model
lr_pipeline.fit(X_train, y_train)

# Extract coefficients from the fitted classifier
lr_model = lr_pipeline.named_steps['classifier']
coefficients = lr_model.coef_[0]
odds_ratios = np.exp(coefficients)

coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient (Beta)': coefficients,
    'Odds_Ratio': odds_ratios,
    'Abs_Impact': np.abs(coefficients)
}).sort_values(by='Abs_Impact', ascending=False).reset_index(drop=True)

print("=== Top 10 Influential Features in Logistic Regression ===")
display(coef_df.head(10))

# Visualize Feature Coefficients
plt.figure(figsize=(10, 8))
top_coefs = pd.concat([coef_df.head(7), coef_df.tail(7)]).drop_duplicates().sort_values(by='Coefficient (Beta)')
colors = ['#E53E3E' if c > 0 else '#38A169' for c in top_coefs['Coefficient (Beta)']]
plt.barh(top_coefs['Feature'], top_coefs['Coefficient (Beta)'], color=colors)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.title('Logistic Regression Feature Coefficients (Log-Odds Impact on Delay)')
plt.xlabel('Coefficient Value (Positive = Increases Delay Risk, Negative = Decreases Risk)')
plt.tight_layout()
plt.show()

=== Top 10 Influential Features in Logistic Regression ===
                   Feature  Coefficient (Beta)  Odds_Ratio  Abs_Impact
0  Weather_Condition_Storm            2.741262   15.506537    2.741262
1   Weather_Condition_Snow            1.788116    5.978180    1.788116
2        Traffic_Level_Low           -1.461753    0.231830    1.461753
3    Weather_Condition_Fog            1.265126    3.543539    1.265126
4     Traffic_Level_Medium           -1.174132    0.309087    1.174132
5   Weather_Condition_Rain            0.862160    2.368270    0.862160
6     Traffic_Level_Severe            0.835403    2.305743    0.835403
7                 Distance            0.392424    1.480566    0.392424
8           Origin_Seattle            0.293770    1.341475    0.293770
9             Waiting_Time            0.265133    1.303605    0.265133


---
## 🌳 Step 4: Model 2 — Decision Tree Classifier Deep Dive

### 1. WHAT We Are Doing
We train a **Decision Tree Classifier** with controlled depth (`max_depth=4`, `min_samples_split=20`, `min_samples_leaf=10`) to ensure white-box interpretability and prevent overfitting. We extract its decision rules, plot the full tree architecture, and analyze Gini feature importance.

### 2. HOW It Works (Mathematical Splitting Criterion)
At each node, the decision tree tests all possible feature split points to find the partition that maximizes **Information Gain** (or maximizes the decrease in **Gini Impurity**).

#### Gini Impurity Formula:
$$I_G(t) = 1 - \sum_{k=1}^{C} p(k|t)^2$$
Where:
- $C$ is the number of classes (here $C=2$: On Time vs. Delayed).
- $p(k|t)$ is the proportion of samples belonging to class $k$ at node $t$.
- $I_G = 0$ signifies a **pure node** (all samples in the node belong to a single class).
- $I_G = 0.5$ represents maximal uncertainty (50% on-time, 50% delayed).

#### Gini Decrease / Feature Importance (MDI):
The overall importance of feature $j$ is the sum of Gini reductions across all nodes split on $j$, weighted by sample proportions:
$$\text{Importance}(j) = \sum_{t \in \text{splits on } j} \frac{N_t}{N} \left[ I_G(t) - \frac{N_{t_L}}{N_t} I_G(t_L) - \frac{N_{t_R}}{N_t} I_G(t_R) \right]$$

### 3. Interview Q&A
> **Interviewer:** *How do you prevent a Decision Tree from overfitting a training dataset?*  
> **Candidate Answer:** *By applying regularization through pre-pruning hyperparameters: (1) Limiting `max_depth` to stop tree growth; (2) Setting `min_samples_split` and `min_samples_leaf` to prevent isolates from forming single-sample leaves; and (3) Post-pruning using Minimal Cost-Complexity Pruning (`ccp_alpha`).*

In [5]:
# Define Decision Tree Pipeline
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=4, 
        min_samples_split=20, 
        min_samples_leaf=10, 
        random_state=42
    ))
])

# Train Decision Tree
dt_pipeline.fit(X_train, y_train)

# Extract fitted classifier
dt_model = dt_pipeline.named_steps['classifier']

# Extract Feature Importances
dt_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': dt_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("=== Top 10 Decision Tree Feature Importances (Gini MDI) ===")
display(dt_importance_df.head(10))

# Visualize Feature Importance
plt.figure(figsize=(10, 5))
top_dt = dt_importance_df.head(8).sort_values(by='Importance', ascending=True)
plt.barh(top_dt['Feature'], top_dt['Importance'], color='#3182CE')
plt.title('Decision Tree Feature Importance (Gini Impurity Reduction)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

# Visualize the Complete Interpretable Decision Tree
plt.figure(figsize=(22, 10))
plot_tree(
    dt_model,
    feature_names=all_feature_names,
    class_names=['On Time', 'Delayed'],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3  # Display top 3 levels for publication clarity
)
plt.title('Explainable Decision Tree Architecture (Top 3 Decision Layers)', fontsize=15, pad=12)
plt.tight_layout()
plt.show()

=== Top 10 Decision Tree Feature Importances (Gini MDI) ===
                   Feature  Importance
0  Weather_Condition_Storm    0.313274
1     Traffic_Level_Severe    0.285901
2             Waiting_Time    0.236624
3                 Distance    0.159811
4            Origin_Dallas    0.004390
5      Transportation_Cost    0.000000
6    Dispatch_Latency_Days    0.000000
7              Cost_Per_KM    0.000000
8        High_Waiting_Flag    0.000000
9           Origin_Chicago    0.000000


---
## ⚖️ Step 5: Rigorous Model Evaluation & Comparison

### 1. WHAT We Are Doing
We evaluate both models on the held-out 20% test set across six critical performance metrics:
1. **Accuracy:** $\frac{TP + TN}{TP + TN + FP + FN}$ (Overall correctness)
2. **Precision:** $\frac{TP}{TP + FP}$ (When model predicts delay, how often is it right?)
3. **Recall (Sensitivity):** $\frac{TP}{TP + FN}$ (Out of all shipments that were delayed, what fraction did we catch?)
4. **F1-Score:** $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ (Harmonic mean balancing precision and recall)
5. **ROC-AUC:** Area under the Receiver Operating Characteristic curve.
6. **Confusion Matrix Analysis:** Assessing the business cost of errors.

### 2. Supply Chain Business Context (Cost of Errors)
- **False Positive (FP):** Model predicts delay, but shipment arrives on time. *Cost: Minor alert fatigue or unnecessary expediting cost.*
- **False Negative (FN):** Model predicts on time, but shipment is actually delayed. *Cost: Severe breach of SLA, customer dissatisfaction, plant shutdowns.*
- **Conclusion:** In logistics, **Recall** is prioritized over pure accuracy because False Negatives carry significantly higher business liabilities.

In [6]:
# Generate Predictions and Probabilities
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]

# Metric Computation Function
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

metrics = [
    evaluate_model('Logistic Regression', y_test, y_pred_lr, y_prob_lr),
    evaluate_model('Decision Tree Classifier', y_test, y_pred_dt, y_prob_dt)
]

eval_df = pd.DataFrame(metrics).set_index('Model')
print("=== Head-to-Head Model Evaluation Comparison Table ===")
display(np.round(eval_df * 100, 2))

# Visual Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Pred: On Time', 'Pred: Delayed'], yticklabels=['Actual: On Time', 'Actual: Delayed'])
axes[0].set_title(f"Logistic Regression Confusion Matrix\nF1: {eval_df.loc['Logistic Regression', 'F1-Score']*100:.1f}%")

cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1], 
            xticklabels=['Pred: On Time', 'Pred: Delayed'], yticklabels=['Actual: On Time', 'Actual: Delayed'])
axes[1].set_title(f"Decision Tree Confusion Matrix\nF1: {eval_df.loc['Decision Tree Classifier', 'F1-Score']*100:.1f}%")

plt.tight_layout()
plt.show()

# Comparative ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {eval_df.loc['Logistic Regression', 'ROC-AUC']:.3f})", color='#3182CE', lw=2)
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {eval_df.loc['Decision Tree Classifier', 'ROC-AUC']:.3f})", color='#38A169', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Chance Baseline')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve Comparison')
plt.legend(loc='lower right')
plt.show()

=== Head-to-Head Model Evaluation Comparison Table ===
                          Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                                   
Logistic Regression          74.23      72.89   68.03     70.38    82.06
Decision Tree Classifier     67.38      67.77   52.48     59.15    71.70


---
## 💾 Step 6: Model & Preprocessor Serialization (Joblib)

### 1. WHAT We Are Doing
We save the trained pipelines and preprocessors to the `models/` directory using `joblib.dump()` so they can be loaded instantly into our Streamlit web application and Power BI automation pipelines.

In [7]:
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(lr_pipeline, os.path.join(models_dir, 'logistic_regression.pkl'))
joblib.dump(dt_pipeline, os.path.join(models_dir, 'decision_tree.pkl'))
joblib.dump(preprocessor, os.path.join(models_dir, 'preprocessor.pkl'))

# Export model evaluation comparison metrics for Power BI
powerbi_dir = '../data/powerbi' if os.path.exists('../data/powerbi') else 'data/powerbi'
os.makedirs(powerbi_dir, exist_ok=True)
eval_df.reset_index().to_csv(os.path.join(powerbi_dir, 'Model_Performance.csv'), index=False)

print(f"Models and preprocessors successfully serialized into '{models_dir}/'")
print(f"Model performance summary saved to '{powerbi_dir}/Model_Performance.csv'")

Models and preprocessors successfully serialized into 'models/'
Model performance summary saved to 'data/powerbi/Model_Performance.csv'


---
## 💡 Step 7: Single Shipment Explainable Prediction Module

### 1. WHAT We Are Doing
We build an **interactive explanation module** that takes any individual shipment manifest and provides:
1. Operational shipment profile.
2. Predicted status & calibrated delay probability.
3. Decision Tree path decomposition (exact if-else branching rules).
4. Top contributing drivers.
5. Plain English executive summary.

### 2. Scientific Integrity Disclaimer
> **⚠️ Critical Interview Note:** Feature importance and regression coefficients quantify statistical association within this dataset; they **do not prove counterfactual causation**. For instance, high transportation cost correlates with delay because longer, congested routes incur detention charges, not because high prices cause trucks to slow down.

In [8]:
def explain_shipment_prediction(sample_input: dict, pipeline=dt_pipeline, lr_pipe=lr_pipeline):
    # Convert input dictionary into DataFrame
    sample_df = pd.DataFrame([sample_input])
    
    # 1. Predictions and probabilities
    pred_class = pipeline.predict(sample_df)[0]
    prob_delay = pipeline.predict_proba(sample_df)[0][1]
    status_label = "Delayed" if pred_class == 1 else "On Time"
    
    # 2. Extract decision path from Decision Tree
    clf = pipeline.named_steps['classifier']
    prep = pipeline.named_steps['preprocessor']
    sample_trans = prep.transform(sample_df)
    
    node_indicator = clf.decision_path(sample_trans)
    leaf_id = clf.apply(sample_trans)[0]
    node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]
    
    rules = []
    for node_id in node_index:
        if leaf_id == node_id:
            continue
        feature_idx = clf.tree_.feature[node_id]
        threshold = clf.tree_.threshold[node_id]
        feature_name = all_feature_names[feature_idx]
        val = sample_trans[0, feature_idx]
        condition = "<=" if val <= threshold else ">"
        rules.append(f"Rule: '{feature_name}' ({val:.2f}) {condition} {threshold:.2f}")
    
    # 3. Print Clean Executive Summary
    print("=" * 65)
    print("📋 EXPLAINABLE SHIPMENT PREDICTION DOSSIER")
    print("=" * 65)
    print(f"Shipment Origin:      {sample_input['Origin']} -> Destination: {sample_input['Destination']}")
    print(f"Distance:             {sample_input['Distance']} km | Shipping Mode: {sample_input['Shipping_Mode']}")
    print(f"Weather Condition:    {sample_input['Weather_Condition']} | Traffic Level: {sample_input['Traffic_Level']}")
    print(f"Dock Waiting Time:    {sample_input['Waiting_Time']} hrs | Unit Cost: ${sample_input['Transportation_Cost']}")
    print("-" * 65)
    print(f"🎯 Predicted Status:   [{status_label.upper()}]")
    print(f"📊 Delay Probability:  {prob_delay * 100:.2f}%")
    print(f"🛡️ Risk Assessment:   {'HIGH RISK OF SLA BREACH' if prob_delay > 0.5 else 'LOW RISK / NORMAL TRANSIT'}")
    print("-" * 65)
    print("🔍 Decision Tree Logic Path:")
    for i, r in enumerate(rules, 1):
        print(f"   Step {i}: {r}")
    print("-" * 65)
    print("📝 Plain English Narrative:")
    if prob_delay > 0.5:
        print(f"Shipment is flagged as DELAYED primarily because adverse conditions (Weather: {sample_input['Weather_Condition']}, "
              f"Traffic: {sample_input['Traffic_Level']}) combined with a dock waiting time of {sample_input['Waiting_Time']} hrs "
              f"exceed the critical transit tolerance threshold.")
    else:
        print(f"Shipment is expected to arrive ON TIME due to favorable environmental conditions and manageable dock waiting time.")
    print("=" * 65)

# Demonstrate with a sample high-risk shipment
sample_shipment = {
    'Origin': 'Chicago',
    'Destination': 'Phoenix',
    'Distance': 2820.0,
    'Shipping_Mode': 'Standard Ground',
    'Weather_Condition': 'Storm',
    'Traffic_Level': 'Severe',
    'Waiting_Time': 8.5,
    'Transportation_Cost': 1450.0,
    'Dispatch_Latency_Days': 2,
    'Cost_Per_KM': 0.514,
    'Distance_Tier': 'Long (>2500 km)',
    'Order_Day_Of_Week': 'Friday',
    'Order_Month': 11,
    'Is_Weekend_Order': 0,
    'High_Waiting_Flag': 1
}

explain_shipment_prediction(sample_shipment)

📋 EXPLAINABLE SHIPMENT PREDICTION DOSSIER
Shipment Origin:      Chicago -> Destination: Phoenix
Distance:             2820.0 km | Shipping Mode: Standard Ground
Weather Condition:    Storm | Traffic Level: Severe
Dock Waiting Time:    8.5 hrs | Unit Cost: $1450.0
-----------------------------------------------------------------
🎯 Predicted Status:   [DELAYED]
📊 Delay Probability:  92.79%
🛡️ Risk Assessment:   HIGH RISK OF SLA BREACH
-----------------------------------------------------------------
🔍 Decision Tree Logic Path:
   Step 1: Rule: 'Weather_Condition_Storm' (1.00) > 0.50
   Step 2: Rule: 'Waiting_Time' (1.76) > -0.81
   Step 3: Rule: 'Distance' (0.36) > -0.98
   Step 4: Rule: 'Origin_Dallas' (0.00) <= 0.50
-----------------------------------------------------------------
📝 Plain English Narrative:
Shipment is flagged as DELAYED primarily because adverse conditions (Weather: Storm, Traffic: Severe) combined with a dock waiting time of 8.5 hrs exceed the critical transit tolera

---
## 🎓 Step 8: Top 10 Technical Interview Questions & Model Answers

### Q1: What is the mathematical difference between Gini Impurity and Entropy in Decision Trees?
**Answer:** Both are impurity functions measuring classification uncertainty.
- Gini Impurity: $I_G = 1 - \sum p_i^2$ (measures the probability of a randomly chosen element being incorrectly labeled).
- Entropy: $H = -\sum p_i \log_2(p_i)$ (information-theoretic measure of bit-level disorder).
In practice, Gini is computationally faster because it avoids calculating logarithms, yielding virtually identical splits in over 98% of industrial applications.

### Q2: Why would you choose a Decision Tree over Logistic Regression?
**Answer:** Decision Trees naturally capture non-linear relationships and high-order feature interactions (e.g., "Storms only cause delays IF shipping mode is Express Air") without requiring manual polynomial feature engineering. Additionally, Decision Trees require no feature scaling and are naturally interpretable as a rule flowchart.

### Q3: Why would you choose Logistic Regression over a Decision Tree?
**Answer:** Logistic Regression provides smooth, well-calibrated probabilities rather than step-function approximations. It is less prone to high variance and small-sample instability, and its feature coefficients offer clear economic elasticity and odds ratios.

### Q4: What is the difference between ROC-AUC and PR-AUC?
**Answer:** ROC-AUC plots True Positive Rate vs False Positive Rate. It can look artificially optimistic on severely imbalanced datasets because a large number of True Negatives suppresses the False Positive Rate. In contrast, the Precision-Recall (PR) curve focuses strictly on the minority positive class without being inflated by True Negatives.

### Q5: What is Data Snooping in feature scaling?
**Answer:** Data snooping occurs when preprocessing transformers (like `StandardScaler`) are fitted on the entire dataset before splitting. This leaks global statistics (such as the test set mean and standard deviation) into training, leading to overly optimistic cross-validation estimates.